# Agent 2 — Garment-Type Classifier (free Colab T4)

Trains a **real Hugging Face vision model** (ViT) to classify garment type from a product photo, then pushes it to `dsreya/garment-type-classifier` on the Hub. The Asli backend then uses this trained model for the `garment_type` field (replacing the ad-hoc VLM read).

**Why not DeepFashion2:** its HF mirrors are gated (Google-Form access), masks-only (no images), or raw-COCO dumps with no usable image+label pairs, and HF Jobs needs paid credits. So we train the same garment-type task on the clean, ungated `ashraq/fashion-product-images-small` (real product images + garment categories).

**How to run:** Runtime → Change runtime type → **T4 GPU**, then Runtime → **Run all**. ~15–25 min. When prompted, paste your Hugging Face **write** token (huggingface.co/settings/tokens).

In [1]:
!pip -q install "transformers>=4.44" "datasets>=2.19" "accelerate>=0.30" evaluate scikit-learn torchvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.8 MB/s eta 0:00:00


In [2]:
from huggingface_hub import login
login()  # paste a WRITE token from https://huggingface.co/settings/tokens

In [3]:
# ---- config ----
HUB_MODEL_ID = "dsreya/garment-type-classifier"   # change the namespace if not dsreya
BASE_MODEL   = "google/vit-base-patch16-224"       # T4-friendly; swap for timm/mobilenetv3_small_100.lamb_in1k for speed
LABEL_COLUMN = "subCategory"                        # garment-type granularity: Topwear / Bottomwear / Dress / ...
MAX_SAMPLES  = 12000                                # subset for a fast, cheap run; raise for higher accuracy
EPOCHS       = 5
import torch; print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only — set Runtime to T4 GPU!")

CUDA: True Tesla T4


In [4]:
# ---- load + prepare the dataset: keep apparel garments, use subCategory as the label ----
from datasets import load_dataset, ClassLabel

ds = load_dataset("ashraq/fashion-product-images-small", split="train")
ds = ds.filter(lambda r: r["masterCategory"] == "Apparel" and r[LABEL_COLUMN])

# Drop ultra-rare classes (need >=2 for a stratified split) and cap size for a quick run.
from collections import Counter
counts = Counter(ds[LABEL_COLUMN])
keep = {k for k, c in counts.items() if c >= 20}
ds = ds.filter(lambda r: r[LABEL_COLUMN] in keep)
if len(ds) > MAX_SAMPLES:
    ds = ds.shuffle(seed=42).select(range(MAX_SAMPLES))

labels = sorted(set(ds[LABEL_COLUMN]))
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for l, i in label2id.items()}
print(len(ds), "images |", len(labels), "classes:", labels)

ds = ds.map(lambda r: {"label": label2id[r[LABEL_COLUMN]]})
ds = ds.cast_column("label", ClassLabel(names=labels))
split = ds.train_test_split(test_size=0.15, seed=42, stratify_by_column="label")
train_ds, eval_ds = split["train"], split["test"]

README.md:   0%|          | 0.00/867 [00:00<?, ?B/s]

data/train-00000-of-00002-6cff4c59f91661(…):   0%|          | 0.00/136M [00:00<?, ?B/s]

data/train-00001-of-00002-bb459e5ac5f01e(…):   0%|          | 0.00/135M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/44072 [00:00<?, ? examples/s]

Filter:   0%|          | 0/44072 [00:00<?, ? examples/s]

Filter:   0%|          | 0/21361 [00:00<?, ? examples/s]

12000 images | 7 classes: ['Apparel Set', 'Bottomwear', 'Dress', 'Innerwear', 'Loungewear and Nightwear', 'Saree', 'Topwear']


Map:   0%|          | 0/12000 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/12000 [00:00<?, ? examples/s]

In [5]:
# ---- processor + transforms ----
from transformers import AutoImageProcessor
import numpy as np

proc = AutoImageProcessor.from_pretrained(BASE_MODEL)
size = proc.size.get("shortest_edge", proc.size.get("height", 224))
mean, std = proc.image_mean, proc.image_std
from torchvision.transforms import (Compose, RandomResizedCrop, Resize, CenterCrop, ToTensor, Normalize, RandomHorizontalFlip)
norm = Normalize(mean=mean, std=std)
train_tf = Compose([RandomResizedCrop(size), RandomHorizontalFlip(), ToTensor(), norm])
eval_tf  = Compose([Resize(size), CenterCrop(size), ToTensor(), norm])

def _apply(tf):
    def f(batch):
        batch["pixel_values"] = [tf(img.convert("RGB")) for img in batch["image"]]
        return batch
    return f
train_ds.set_transform(_apply(train_tf))
eval_ds.set_transform(_apply(eval_tf))

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/69.7k [00:00<?, ?B/s]

In [6]:
# ---- model + Trainer ----
import torch, evaluate, numpy as np
from transformers import AutoModelForImageClassification, TrainingArguments, Trainer

model = AutoModelForImageClassification.from_pretrained(
    BASE_MODEL, num_labels=len(labels), id2label=id2label, label2id=label2id,
    ignore_mismatched_sizes=True,
)

def collate(batch):
    return {"pixel_values": torch.stack([b["pixel_values"] for b in batch]),
            "labels": torch.tensor([b["label"] for b in batch])}

acc = evaluate.load("accuracy")
def metrics(p):
    return acc.compute(predictions=np.argmax(p.predictions, axis=1), references=p.label_ids)

args = TrainingArguments(
    output_dir="garment-type-classifier",
    per_device_train_batch_size=32, per_device_eval_batch_size=32,
    num_train_epochs=EPOCHS, learning_rate=5e-5, warmup_steps=100,
    eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
    load_best_model_at_end=True, metric_for_best_model="accuracy", greater_is_better=True,
    logging_steps=25, remove_unused_columns=False, fp16=torch.cuda.is_available(),
    push_to_hub=True, hub_model_id=HUB_MODEL_ID, report_to="none",
)
# transformers 5.x: pass the processor as processing_class (the old `tokenizer=` arg was removed).
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=eval_ds,
                  data_collator=collate, compute_metrics=metrics, processing_class=proc)
trainer.train()
print("eval:", trainer.evaluate())

[transformers] You passed `num_labels=7` which is incompatible to the `id2label` map of length `1000`.


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                          
------------------+----------+------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([7, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([7])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


Epoch,Training Loss,Validation Loss,Accuracy
1,0.211411,0.117696,0.963889
2,0.155393,0.086293,0.971111
3,0.096036,0.077816,0.974444
4,0.097119,0.065603,0.978333
5,0.074458,0.071122,0.978889


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['vit.layers.0.attention.q_proj.weight', 'vit.layers.0.attention.q_proj.bias', 'vit.layers.0.attention.k_proj.weight', 'vit.layers.0.attention.k_proj.bias', 'vit.layers.0.attention.v_proj.weight', 'vit.layers.0.attention.v_proj.bias', 'vit.layers.0.attention.o_proj.weight', 'vit.layers.0.attention.o_proj.bias', 'vit.layers.0.layernorm_before.weight', 'vit.layers.0.layernorm_before.bias', 'vit.layers.0.layernorm_after.weight', 'vit.layers.0.layernorm_after.bias', 'vit.layers.0.mlp.fc1.weight', 'vit.layers.0.mlp.fc1.bias', 'vit.layers.0.mlp.fc2.weight', 'vit.layers.0.mlp.fc2.bias', 'vit.layers.1.attention.q_proj.weight', 'vit.layers.1.attention.q_proj.bias', 'vit.layers.1.attention.k_proj.weight', 'vit.layers.1.attention.k_proj.bias', 'vit.layers.1.attention.v_proj.weight', 'vit.layers.1.attention.v_proj.bias', 'vit.layers.1.attention.o_proj.weight', 'vit.layers.1.attention.o_proj.bias', 'vit.layers.1.layernorm_before

Training Loss,Validation Loss,Epoch,Accuracy
0.074458,0.071122,5,0.978889


eval: {'eval_loss': 0.07112240046262741, 'eval_accuracy': 0.9788888888888889}


In [7]:
# ---- push the trained model + processor to the Hub ----
trainer.push_to_hub()
proc.push_to_hub(HUB_MODEL_ID)
print("pushed to https://huggingface.co/" + HUB_MODEL_ID)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

README.md:   0%|          | 0.00/1.83k [00:00<?, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.


pushed to https://huggingface.co/dsreya/garment-type-classifier


In [8]:
# ---- quick sanity inference ----
from transformers import pipeline
clf = pipeline("image-classification", model=HUB_MODEL_ID)
print(clf(eval_ds[0]["image"]))

config.json:   0%|          | 0.00/978 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/343M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/325 [00:00<?, ?B/s]

[{'label': 'Topwear', 'score': 0.9997499585151672}, {'label': 'Bottomwear', 'score': 0.00011219733278267086}, {'label': 'Dress', 'score': 8.243112824857235e-05}, {'label': 'Apparel Set', 'score': 2.8755966923199594e-05}, {'label': 'Innerwear', 'score': 1.1757861102523748e-05}]
